In [11]:
import os
import json
import glob

import numpy as np
import pandas as pd

In [12]:
runs = [x for x in sorted(glob.glob('pinn_data/*/*/*')) if 'old' not in x]
runs

['pinn_data/maxwell_bdy/donut/seed_1',
 'pinn_data/maxwell_bdy/donut/seed_2',
 'pinn_data/maxwell_bdy/donut/seed_3',
 'pinn_data/maxwell_bdy/donut/seed_4',
 'pinn_data/maxwell_bdy/donut/seed_5',
 'pinn_data/maxwell_bdy/hopf/seed_1',
 'pinn_data/maxwell_bdy/hopf/seed_2',
 'pinn_data/maxwell_bdy/hopf/seed_3',
 'pinn_data/maxwell_bdy/hopf/seed_4',
 'pinn_data/maxwell_bdy/hopf/seed_5',
 'pinn_data/maxwell_bdy/planewave/seed_1',
 'pinn_data/maxwell_bdy/planewave/seed_2',
 'pinn_data/maxwell_bdy/planewave/seed_3',
 'pinn_data/maxwell_bdy/planewave/seed_4',
 'pinn_data/maxwell_bdy/planewave/seed_5',
 'pinn_data/maxwell_bdy/random/seed_1',
 'pinn_data/maxwell_bdy/random/seed_2',
 'pinn_data/maxwell_bdy/random/seed_3',
 'pinn_data/maxwell_bdy/random/seed_4',
 'pinn_data/maxwell_bdy/random/seed_5',
 'pinn_data/maxwell_nbdy/donut/seed_1',
 'pinn_data/maxwell_nbdy/donut/seed_2',
 'pinn_data/maxwell_nbdy/donut/seed_3',
 'pinn_data/maxwell_nbdy/donut/seed_4',
 'pinn_data/maxwell_nbdy/donut/seed_5',


In [13]:
df = pd.DataFrame(columns=['add_bc', 'soln_name', 'seed', 'min_best_val_rel_err', 'time_to_min_best_val_rel_err', 'time_to_5p_best_val_rel_err', 'residual'])

In [14]:
for run in runs:
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    log = log.sort_values('epoch').reset_index(drop=True)
    log['residual'] = np.sqrt((6*log['best_val_pde_loss'] + 2*log['best_val_div_loss'] ) / 8)
    log['rel_l2_error'] = log['rel_l2_error'] * 100.0
    log = log.round(2)
    
    info = {}
    
    split = run.split(os.path.sep)
    
    info['add_bc'] = True if '_bdy' in split[1] else False
    info['soln_name'] = split[2]
    info['seed'] = split[3].replace('seed_', '')
    
    # work in fraction-space throughout; only convert to percent for the final stored value
    min_best_val_rel_err_frac = log['rel_l2_error'].min()
    info['min_best_val_rel_err'] = min_best_val_rel_err_frac
    
    # epoch where we reach minimum validation relative L2 err
    min_best_val_rel_err_epoch = int(log[log['rel_l2_error'] <= min_best_val_rel_err_frac].iloc[0]['epoch'])
    info['time_to_min_best_val_rel_err'] = log[log['epoch'] <= min_best_val_rel_err_epoch]['training_time'].iloc[-1].item()
    
    # epoch where we first reach 5% relative err (0.05 in fraction-space)
    try:
        first_5p_epoch = int(log[log['rel_l2_error'] < 5.0].iloc[0]['epoch'])
        info['time_to_5p_best_val_rel_err'] = log[log['epoch'] <= first_5p_epoch]['training_time'].iloc[-1].item()
        info['residual'] = log['residual'][log['epoch'] == first_5p_epoch].item()
    except IndexError:
        info['time_to_5p_best_val_rel_err'] = np.nan
        info['residual'] = log['residual'][log['epoch'] == min_best_val_rel_err_epoch].item()
    
    df.loc[len(df)] = info

In [15]:
df['seed'] = df['seed'].astype(float)

results = df[[
    'soln_name',
    'time_to_5p_best_val_rel_err',
    'time_to_min_best_val_rel_err',
    'min_best_val_rel_err',
    'add_bc',
    'seed',
    'residual'
]].rename(columns={
    'time_to_5p_best_val_rel_err': 'Time to <5% RL2E. (s)',
    'time_to_min_best_val_rel_err': 'Time to min. RL2E. (s)',
    'min_best_val_rel_err': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC',
    'residual': 'Residual'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative err columns
        display_results[col] = mean.map('{:.2f}'.format) + ' ± ' + sem.map('{:.2f}'.format)

display_results = pd.DataFrame(display_results).reset_index()
display_results['Solution'] = display_results['Solution'].map(
    {'random': 4, 'planewave': 1, 'donut': 2, 'hopf': 3}
)
display_results = display_results.sort_values(by=['Solution', 'w/ BC'], ascending=[True, False])
display_results = display_results[['Solution', 'w/ BC', 'Min. RL2E. (%)', 'Time to <5% RL2E. (s)', 'Time to min. RL2E. (s)', 'Residual']].reset_index(drop=True)
display_results

,Solution,w/ BC,Min. RL2E. (%),Time to <5% RL2E. (s),Time to min. RL2E. (s),Residual
0,1,True,1.00 ± 0.11,82.9 ± 6.8,259.8 ± 29.2,0.03 ± 0.00
1,1,False,3.22 ± 2.14,69.2 ± 9.0,124.5 ± 8.9,0.03 ± 0.00
2,2,True,95.97 ± 2.47,NaN,12.8 ± 7.9,0.09 ± 0.02
3,2,False,86.70 ± 0.57,NaN,56.4 ± 15.3,0.07 ± 0.03
4,3,True,2.43 ± 0.04,59.5 ± 1.9,263.8 ± 51.7,0.01 ± 0.00
5,3,False,0.88 ± 0.04,6.0 ± 1.8,78.4 ± 15.0,0.03 ± 0.00
6,4,True,5.11 ± 0.20,331.0 ± 41.4,384.5 ± 16.3,0.02 ± 0.00
7,4,False,2.42 ± 0.24,42.7 ± 6.9,148.2 ± 24.8,0.02 ± 0.00


In [17]:
print(display_results.to_markdown(index=False))

|   Solution | w/ BC   | Min. RL2E. (%)   | Time to <5% RL2E. (s)   | Time to min. RL2E. (s)   | Residual    |
|-----------:|:--------|:-----------------|:------------------------|:-------------------------|:------------|
|          1 | True    | 1.00 ± 0.11      | 82.9 ± 6.8              | 259.8 ± 29.2             | 0.03 ± 0.00 |
|          1 | False   | 3.22 ± 2.14      | 69.2 ± 9.0              | 124.5 ± 8.9              | 0.03 ± 0.00 |
|          2 | True    | 95.97 ± 2.47     | nan                     | 12.8 ± 7.9               | 0.09 ± 0.02 |
|          2 | False   | 86.70 ± 0.57     | nan                     | 56.4 ± 15.3              | 0.07 ± 0.03 |
|          3 | True    | 2.43 ± 0.04      | 59.5 ± 1.9              | 263.8 ± 51.7             | 0.01 ± 0.00 |
|          3 | False   | 0.88 ± 0.04      | 6.0 ± 1.8               | 78.4 ± 15.0              | 0.03 ± 0.00 |
|          4 | True    | 5.11 ± 0.20      | 331.0 ± 41.4            | 384.5 ± 16.3             | 0.02 ± 0.00 |
|